# Checking my own subject list against what else CDA knows

*Standalone -- no prior vignette required.*

> **DRAFT STATUS:** specific match counts below have not yet been run against the live API. The file (`cases.tsv`) and the `upstream_id` match target are confirmed real.

I pulled a case list straight out of GDC -- demographics, diagnoses, project info, which experimental strategies each case has, file counts, all of it. This is the file I've actually been working in. I'm not looking for new subjects, and I'm not trying to explore anything -- I already have my cohort. What I want to know is whether there's *more* data about these specific people sitting in CDA that GDC alone doesn't show me, without retyping every case ID into a filter by hand, and without making a separate clean file just for CDA -- I want to use the file I already have.

In [ ]:
from cdapython import *
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

My file, `cases.tsv`, has a column called `id` with GDC's own case UUIDs in it -- `bc84c5c5-1785-4edd-b732-8987f862063e` and so on. These aren't CDA's internal subject IDs; they're the native identifiers GDC itself assigned, the same ones I'd already be using in any GDC-side analysis. `match_from_file` takes three things: which file, which column in that file to use, and which CDA column to match it against. Since I'm matching native source IDs instead of CDA's own IDs, I match against `upstream_id`, not `subject_id`:

In [ ]:
get_subject_data(
    match_from_file={
        'input_file': 'cases.tsv',
        'input_column': 'id',
        'cda_column_to_match': 'upstream_id'
    },
    add_columns='upstream_identifiers.*'
)

This matters because I never had to convert anything -- the exact IDs sitting in my existing GDC export work directly as a CDA lookup key, as long as `upstream_id` is what I tell it to match against instead of CDA's own ID scheme. Adding `upstream_identifiers.*` shows me exactly which sources have data on each case -- these are all TCGA-OV subjects pulled from GDC, so I'm curious whether any of them also show up in PDC or another source I wasn't already looking at.

Before pulling full records, I want a shape of what I'd actually be getting:

In [ ]:
summarize_files(match_from_file={
    'input_file': 'cases.tsv',
    'input_column': 'id',
    'cda_column_to_match': 'upstream_id'
})

In [ ]:
summarize_subjects(match_from_file={
    'input_file': 'cases.tsv',
    'input_column': 'id',
    'cda_column_to_match': 'upstream_id'
})

Now that I know whether there's more data out there on these cases than GDC alone showed me, I want to know exactly where each piece came from -- not just "PDC has something," but a pointer back to that source's own record for this specific case, so I can look at it directly or cite it properly:

In [ ]:
subjects_with_refs = get_subject_data(
    match_from_file={
        'input_file': 'cases.tsv',
        'input_column': 'id',
        'cda_column_to_match': 'upstream_id'
    },
    include_external_refs=True
)
subjects_with_refs

There's a new `external_reference_data` column here -- a small table per subject pointing back to the actual upstream record that described them, separate from `data_source` (which just tells me *which* data center a row came from, not the specific record). This is the kind of thing I'd want before citing a source properly in a publication, or before trusting that CDA's version of a record matches what GDC's own record actually says.

And my case file -- demographics, diagnosis fields, experimental strategies, file counts, all of it -- never had to leave my spreadsheet in the first place. I searched using the export I already had, not a stripped-down copy made just for this lookup.